In [78]:
import pandas as pd 
from pathlib import Path
import re 
import json 

In [50]:
p = Path("/home/finn/workspace/creatures/logs/simulation_1782553328.log")
assert p.exists()

In [64]:
with open(p, "r") as f:
    lines = f.readlines()

print(len(lines))

1425


In [65]:
lines

['[simulation_start_ts=1782553328] [frame=0] [level=INFO] simulation_start start_ts=1782553328 seed=1234\n',
 '[simulation_start_ts=1782553328] [frame=0] [level=DEBUG] plant_spawn source=startup x=-304.79 y=-370.67\n',
 '[simulation_start_ts=1782553328] [frame=0] [level=DEBUG] plant_spawn source=startup x=466.77 y=173.09\n',
 '[simulation_start_ts=1782553328] [frame=0] [level=DEBUG] plant_spawn source=startup x=-262.47 y=322.71\n',
 '[simulation_start_ts=1782553328] [frame=0] [level=DEBUG] plant_spawn source=startup x=590.12 y=-122.58\n',
 '[simulation_start_ts=1782553328] [frame=0] [level=DEBUG] plant_spawn source=startup x=56.20 y=-157.97\n',
 '[simulation_start_ts=1782553328] [frame=0] [level=DEBUG] plant_spawn source=startup x=347.50 y=-261.60\n',
 '[simulation_start_ts=1782553328] [frame=0] [level=DEBUG] plant_spawn source=startup x=169.47 y=-46.04\n',
 '[simulation_start_ts=1782553328] [frame=0] [level=DEBUG] plant_spawn source=startup x=409.95 y=-333.21\n',
 '[simulation_start_t

In [66]:
def extract_animal_data(line):
    m = re.match(
        r".*animal_despawn,reason=(?P<reason>.*),spawn_at_frame=(?P<spawn_at>.*),despawn_at_frame=(?P<despawn_at>.*),lifetime_frames=(?P<lifetime>.*),genome=(?P<genome>.*),diet=(?P<diet>.*).*",
        line,
    )

    if m is not None:
        d = {}
        d["reason"] = m.group("reason")
        d["spawn_at"] = float(m.group("spawn_at"))
        d["despawn_at"] = float(m.group("despawn_at"))
        d["lifetime"] = float(m.group("lifetime"))
        d["genome"] = m.group("genome")
        d["diet"] = m.group("diet")
        return d
    return None


In [67]:
df = pd.DataFrame(list(filter(None, map(extract_animal_data, lines))))
df

,reason,spawn_at,despawn_at,lifetime,genome,diet
0,starvation,291.0,905.0,614.0,"[-0.723959, 0.9796271, -0.66339016, -0.4011550...",Herbivore
1,starvation,0.0,1072.0,1072.0,"[-0.76308155, 0.9923005, -0.66070104, -0.37003...",Herbivore
2,starvation,630.0,1248.0,618.0,"[-0.7507587, 1.0269265, -0.6788975, -0.3622777...",Herbivore
3,starvation,630.0,1304.0,674.0,"[-0.72461605, 1.0208873, -0.62558854, -0.42920...",Herbivore


In [143]:
def extract_population_data(line):
    m = re.match(
        r".*frame=(?P<frame>d*).*population_size plants=(?P<n_plants>.*) animals=AnimalPopulation (?P<json>\{.*\}).*",
        line,
    )
    if m is not None:
        d = {}
        #d["frame"] = int(m.group("frame"))
        d["n_plants"] = int(m.group("n_plants"))
        json_data = json.loads(m.group("json"))
        d["carnivores"] = int(json_data.get("carnivores", 0))
        d["herbivores"] = int(json_data.get("herbivores", 0))
        d["omnivores"] = int(json_data.get("omnivores", 0))
        return d
    return m     

In [144]:
df = pd.DataFrame(list(filter(None, map(extract_population_data, lines))))
df

JSONDecodeError: Expecting property name enclosed in double quotes: line 1 column 3 (char 2)

In [ ]:
2026-06-27T09:42:30.541231Z  INFO bevy_project::logging: [frame=1301] population_size plants=43 animals=AnimalPopulation { carnivores: 1, herbivores: 12, omnivores: 0 }